# Fresh reviewed retrieval benchmark v1

48 cases / 12 paired bilingual intents, not 48 independent questions. Select a GPU runtime and run cells in order.
Upload `fresh_reviewed_colab_inputs.zip` when prompted. No GitHub clone, API key or paid inference API is required.

Pinned E5 embeds the 24 NEW query texts on Colab. Existing 1,063 document vectors are reused after hash/order checks.
All five target-language documents compete with the existing admin scope. Freeze top30 before BGE reranking.
Compare dense top10 and three reranking inputs: plain, parent-plain, narrative-parent, final10.
Use FP32 on the same GPU. Identical model inputs reuse exact new-run scores; narrative-parent uses plain table
scores and predecessor narrative scores, with verified lineage. Shared-compute timings are not independent arm latency.

Labels are assistant-prepared/reviewed and user-approved, not independent held-out gold. Added preceding context is
ranking-only; evidence is scored on ORIGINAL selected anchors. No translation layer or LLM answer generation.
Models download ONLY on Colab. Per-input limits are checked; no silent truncation and no total context budget.


In [ ]:
%pip -q install sentence-transformers==6.0.1 transformers==5.17.0 huggingface-hub==1.31.0 numpy==2.4.6 python-dotenv
# If asked to restart, restart and continue with the next cell.


In [ ]:
import os
os.environ.update(OMP_NUM_THREADS='4', MKL_NUM_THREADS='4', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then rerun.'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()  # Select fresh_reviewed_colab_inputs.zip.
assert 'fresh_reviewed_colab_inputs.zip' in uploaded, 'Select the input ZIP, not the notebook.'
work=Path('/content/fresh_reviewed_experiment')
assert not work.exists(), 'Inputs already extracted; continue below or start a fresh runtime.'
work.mkdir()
with zipfile.ZipFile('fresh_reviewed_colab_inputs.zip') as bundle:
    assert all((work/name).resolve().is_relative_to(work.resolve()) for name in bundle.namelist())
    bundle.extractall(work)
del uploaded
os.chdir(work)


In [ ]:
import subprocess,sys
runner=[sys.executable,'scripts/retrieval/evaluate_fresh_reviewed_reranker.py']
# No models loaded/downloaded in this cell; validates package, labels and saved vectors.
subprocess.run(runner+['--preflight'],check=True)


In [ ]:
output=Path('/content/fresh_reviewed_results')
# Downloads pinned E5/BGE models to Colab, encodes new queries, freezes candidate pools and reranks.
# Completed/partial output folders are protected. A repeat needs a NEW output path.
subprocess.run(runner+['--output-dir',str(output),'--device','cuda','--allow-download'],check=True)


In [ ]:
# Offline replay: checks all scores, rendered inputs, rankings and evidence character coverage.
subprocess.run(runner+['--output-dir',str(output),'--verify-only'],check=True)
import json
report=json.loads((output/'results.json').read_text())
print('Direction | Dense | Plain | Parent | Narrative | Candidate30 availability')
for direction in ('vi->vi','en->en','vi->en','en->vi'):
    counts=[]
    for arm in ('dense_top10','plain','parent-plain','narrative-parent'):
        s=report['summary'][arm][direction]
        counts.append(f"{s['full_evidence_cases_at_10']}/{s['full_intent_eligible_cases']}")
    s=report['summary']['candidate_pool30'][direction]
    counts.append(f"{s['full_evidence_cases_at_30']}/{s['full_intent_eligible_cases']}")
    print(direction,*counts,sep=' | ')


In [ ]:
# Download ONLY outputs, including partial checkpoints if an earlier cell failed.
import shutil
assert output.exists(), 'Run the benchmark cell first.'
archive=shutil.make_archive('/content/fresh_reviewed_results','zip',root_dir=output)
files.download(archive)
# Return fresh_reviewed_results.zip to Codex for independent replay and gains/losses analysis.
# A failed/partial config is NOT a completed benchmark.
